# Counterfactual 1-hop questions: our decoder vs SPARKLE

Runs the hand-selected counterfactual questions in `counterfactuals/counterfactual 1 hops.json` through two systems, and scores each output against the counterfactual SPARQL query, which has no answer in DBpedia 2016-04:

- **Ours**: the fine-tuned Qwen2.5-Coder-1.5B with positive ontological guidance (`generate_positive`) at beam width 3.
- **SPARKLE** ([zzaebok/SPARKLE](https://github.com/zzaebok/SPARKLE)): its released LC-QuAD 1.0 BART-large checkpoint under its own KG-constrained decoding, at its own LC-QuAD setting (10 beams).

The counterfactuals are all SELECT queries. SPARKLE's decoder (`sparkle/sparql_token_generator.py`) only checks that each entity and relation of an ASK query exists, so an ASK counterfactual is as reachable for SPARKLE as for us and cannot tell the two apart. A SELECT query, though, may only use a relation that SPARKLE's copy of DBpedia already links to the subject entity. No counterfactual subject is linked to its relation, so SPARKLE cannot generate any of them.

**Before running:**
1. The Drive folder `MyDrive/ogd/` must hold `qwen_lcquad.safetensors` and `class_tries.pkl` (the same two files `run_eval_colab.ipynb` uses).
2. Runtime → Change runtime type → GPU, with **High-RAM** if available. SPARKLE keeps DBpedia's entity–relation links in memory to constrain decoding, which may need more than a standard runtime's RAM.

Then Runtime → Run all. Each system writes its outputs to `MyDrive/ogd/counterfactuals/`, and is skipped on a re-run while that file holds exactly the current questions; a file for a different set of questions is regenerated. If SPARKLE runs out of memory, switch to High-RAM and run all again: ours will not be regenerated.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)  # re-running this cell fixes dropped Drive connections

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ogd')   # where run_eval_colab.ipynb's two files live
OUT_DIR = DRIVE_DIR / 'counterfactuals'           # both systems' outputs and the comparison land here
REPO_DIR = Path('/content/ontologically-guided-decoding')
OGD_BRANCH = 'counterfactuals'   # the counterfactuals folder is only on this branch until it is merged into main
SPARKLE_DIR = Path('/content/SPARKLE')
SPARKLE_DATA = Path('/content/sparkle_data')     # SPARKLE's released LC-QuAD 1.0 checkpoint and KG files

for name in ('qwen_lcquad.safetensors', 'class_tries.pkl'):
    assert (DRIVE_DIR / name).exists(), f'{name} not found in {DRIVE_DIR}'
OUT_DIR.mkdir(exist_ok=True)

Mounted at /content/drive


In [2]:
!git clone -q https://github.com/JosephMuddle/ontologically-guided-decoding.git {REPO_DIR} 2>/dev/null; git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q {OGD_BRANCH} && git -C {REPO_DIR} pull -q
!git clone -q https://github.com/zzaebok/SPARKLE.git {SPARKLE_DIR} 2>/dev/null; git -C {SPARKLE_DIR} pull -q
!git -C {REPO_DIR} log --oneline -1
!git -C {SPARKLE_DIR} log --oneline -1

55c23ab (HEAD -> counterfactuals, origin/counterfactuals) Restrict the counterfactuals to SELECT queries
2c66560 (HEAD -> master, origin/master, origin/HEAD) Update README.md


## Our decoder

`run_ogd.py` runs in its own process, so its GPU memory and its tries are released before SPARKLE loads.

In [3]:
import shutil
(REPO_DIR / 'model').mkdir(exist_ok=True)
(REPO_DIR / 'dbpedia').mkdir(exist_ok=True)
for src, dst in ((DRIVE_DIR / 'qwen_lcquad.safetensors', REPO_DIR / 'model' / 'qwen_lcquad.safetensors'),
                 (DRIVE_DIR / 'class_tries.pkl', REPO_DIR / 'dbpedia' / 'class_tries.pkl')):
    if not dst.exists():
        shutil.copy(src, dst)
print('weights and tries on VM local disk')

# .env holds only paths here (no secrets); config and tokenizer come from the
# public base model Qwen/Qwen2.5-Coder-1.5B, only the weights are local
(REPO_DIR / '.env').write_text('MODEL_WEIGHTS=model/qwen_lcquad.safetensors\nDATA_PATH=dbpedia\n')
!pip install -q xgrammar safetensors

weights and tries on VM local disk


In [4]:
%%writefile /content/run_ogd.py
"""Our decoder on the counterfactual 1-hop questions: generate_positive() -- the
hard constraints plus positive ontological guidance -- at beam width 3.

    python run_ogd.py <repo dir> <output json>
"""
import json
import os
import sys
import time
from pathlib import Path

repo_dir, out_path = Path(sys.argv[1]), Path(sys.argv[2])
data = json.loads((repo_dir / "counterfactuals" / "counterfactual 1 hops.json").read_text(encoding="utf-8"))
# kept when it holds exactly these questions, replaced when they have changed
if out_path.exists():
    stored = [r["question"] for r in json.loads(out_path.read_text(encoding="utf-8"))]
    if stored == [item["question"] for item in data]:
        print(f"{out_path} is up to date -- delete it to regenerate")
        sys.exit(0)
    print(f"{out_path} holds a different set of questions -- regenerating it", flush=True)

# read once, when the generation module is imported, as test.py's --beams does
os.environ["BEAM_WIDTH"] = "3"
sys.path.insert(0, str(repo_dir))
import type_constrained_generation as tcg  # noqa: E402 -- loads weights, grammars, tries

assert tcg.BEAM_WIDTH == 3, f"beam width is {tcg.BEAM_WIDTH}, not 3"

results = []
start = time.perf_counter()
for i, item in enumerate(data, 1):
    try:
        produced = tcg.generate_positive(item["question"])
    except Exception as e:  # one bad question must not lose the others
        produced = f"ERROR: {e}"
    results.append({"question": item["question"], "gold": item["sparql_query"], "generated": produced})
    print(f"{i}/{len(data)}  {produced}", flush=True)

out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{len(results)} questions in {time.perf_counter() - start:.0f}s -> {out_path}")

Overwriting /content/run_ogd.py


In [5]:
!python /content/run_ogd.py {REPO_DIR} {OUT_DIR}/ogd_beam3_positive.json

/content/drive/MyDrive/ogd/counterfactuals/ogd_beam3_positive.json holds a different set of questions -- regenerating it
loaded qwen_lcquad.safetensors into Qwen/Qwen2.5-Coder-1.5B on cuda
compiled grammars: 5 beginning templates, 616 relations + type tail, 763 classes
loaded 762 class tries in 16.1s
1/23  SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Roberto_Clemente_Bridge> <http://dbpedia.org/ontology/routeEnd> ?uri }
2/23  SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Birmingham> <http://dbpedia.org/ontology/nearestCity> ?uri . <http://dbpedia.org/resource/Oxford_Junction,_Iowa> <http://dbpedia.org/ontology/nearestCity> ?uri }
3/23  SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Coffee> <http://dbpedia.org/ontology/ingredient> ?uri }
4/23  SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Demis_Roussos> <http://dbpedia.org/ontology/training> ?uri }
5/23  SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Aishath_Saffa> <http://dbped

## SPARKLE

SPARKLE pins torch 1.13 and transformers 4.30 for Python 3.8, which Colab cannot install, so its code runs against Colab's own torch and transformers. Two changes in transformers since 4.30 are handled in `run_sparkle.py` without editing SPARKLE's code (its docstring says how), and the README's own quickstart example is decoded first as a check that SPARKLE still behaves as published.

The download is SPARKLE's `lcquad1.zip` (2.2 GB, about 4 GB unpacked).

In [6]:
ckpt = SPARKLE_DATA / 'lcquad1' / 'output' / 'pytorch_model.bin'
if not ckpt.exists():
    !wget -q -O /content/lcquad1.zip https://storage.googleapis.com/sparkle-code/lcquad1.zip
    !unzip -q -o /content/lcquad1.zip -d {SPARKLE_DATA}
    !rm -f /content/lcquad1.zip
print(sorted(p.name for p in (SPARKLE_DATA / 'lcquad1' / 'preprocessed').iterdir()))

# only what SPARKLE needs and Colab lacks: marisa-trie for its tries,
# SPARQLWrapper because its LC-QuAD module imports it
!pip install -q marisa-trie SPARQLWrapper

['entity_relation_token_dict.pkl', 'entity_relation_tries.pkl', 'eval_data.json', 'link_dictionaries.pkl', 'test_data.json', 'train_data.json', 'train_eval_data.json']


In [7]:
%%writefile /content/run_sparkle.py
"""SPARKLE (Lee & Shin, 2024) on the counterfactual 1-hop questions.

Its released LC-QuAD 1.0 checkpoint, decoded under its own constrained decoding
(SparqlTokenGenerator) at its own LC-QuAD setting: tasks.py's generate-lcquad1
uses 10 beams, returns all 10, and stops at length 128.

SPARKLE was written against transformers 4.30. Two things have changed under it
since, and both are put back here without touching its code:
  forced_bos_token_id  newer transformers no longer keep generation settings on
      the model config, but SparqlTokenGenerator reads this one there to know
      where the query starts. It is handed the value generate() actually uses.
  empty candidate lists  4.30 turned an empty prefix_allowed_tokens_fn result
      into an all -inf row, which retires that beam; newer versions raise an
      error instead. SparkleConstraint applies SPARKLE's function the 4.30 way.

    python run_sparkle.py <SPARKLE repo> <lcquad1 data dir> <our repo> <output json>
"""
import json
import math
import pickle
import sys
import time
from pathlib import Path
from types import SimpleNamespace

sparkle_dir, data_dir, repo_dir, out_path = map(Path, sys.argv[1:5])
data = json.loads((repo_dir / "counterfactuals" / "counterfactual 1 hops.json").read_text(encoding="utf-8"))
# kept when it holds exactly these questions, replaced when they have changed
if out_path.exists():
    stored = [r["question"] for r in json.loads(out_path.read_text(encoding="utf-8"))]
    if stored == [item["question"] for item in data]:
        print(f"{out_path} is up to date -- delete it to regenerate")
        sys.exit(0)
    print(f"{out_path} holds a different set of questions -- regenerating it", flush=True)

import torch  # noqa: E402
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer,  # noqa: E402
                          LogitsProcessor, LogitsProcessorList)

sys.path.insert(0, str(sparkle_dir))
from sparkle.sparql_token_generator import SparqlTokenGenerator  # noqa: E402
from sparkle.utils import UniqueToken, encode_fn  # noqa: E402

NUM_BEAMS = 10   # tasks.py, generate-lcquad1
MAX_LENGTH = 128  # ditto
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

ckpt = data_dir / "output"
tokenizer = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSeq2SeqLM.from_pretrained(ckpt).to(DEVICE).eval()
# SPARKLE passes no_repeat_ngram_size=None to switch off the checkpoint's 3;
# 0 is the value every transformers version reads as off
model.generation_config.no_repeat_ngram_size = 0
forced_bos = model.generation_config.forced_bos_token_id
print(f"loaded {ckpt} on {DEVICE}, forced BOS token {forced_bos}", flush=True)

t0 = time.time()
with open(data_dir / "preprocessed" / "entity_relation_tries.pkl", "rb") as f:
    entity_trie, relation_trie = pickle.load(f)
with open(data_dir / "preprocessed" / "link_dictionaries.pkl", "rb") as f:
    head_rel, rel_tail = pickle.load(f)
print(f"loaded tries and link dictionaries in {time.time() - t0:.0f}s", flush=True)

# built exactly as generate_sparkle.py builds it, with generate-lcquad1's tokens
sparql_vocab = {
    "SELECT": encode_fn("SELECT DISTINCT ", tokenizer),
    "ASK": encode_fn("ASK ", tokenizer),
    "VARS": [encode_fn(token, tokenizer) for token in ("var_uri ", "var_x ")],
}
for token in UniqueToken:
    sparql_vocab[token.name] = encode_fn(token.value, tokenizer)
print(sparql_vocab, flush=True)

t0 = time.time()
generator = SparqlTokenGenerator(
    tokenizer=tokenizer,
    entity_trie=entity_trie,
    relation_trie=relation_trie,
    head_rel=head_rel,
    relation_tail=rel_tail,
    sparql_vocab=sparql_vocab,
    config=SimpleNamespace(forced_bos_token_id=forced_bos),
)
print(f"built SPARKLE's decoder in {time.time() - t0:.0f}s", flush=True)


class SparkleConstraint(LogitsProcessor):
    """SPARKLE's prefix_allowed_tokens_fn, applied as transformers 4.30 applied it:
    every token outside the allowed list is set to -inf, and an empty list leaves
    the whole row at -inf. On the first step the forced-BOS processor, which runs
    before this one, has already left only BOS; 4.30 ran it after the prefix mask,
    where it overrode that mask, so that step is passed through untouched."""

    def __init__(self, allowed_fn, num_beams, forced_bos_token_id):
        self.allowed_fn = allowed_fn
        self.num_beams = num_beams
        self.forced_bos_token_id = forced_bos_token_id

    def __call__(self, input_ids, scores):
        if self.forced_bos_token_id is not None and input_ids.shape[-1] == 1:
            return scores
        mask = torch.full_like(scores, -math.inf)
        for row, sent in enumerate(input_ids):
            allowed = self.allowed_fn(row // self.num_beams, sent)
            if len(allowed):
                mask[row, allowed] = 0
        return scores + mask


def generate(text, num_beams):
    """All num_beams candidates for one question, best first."""
    inputs = tokenizer([text], max_length=256, return_tensors="pt",
                       padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            logits_processor=LogitsProcessorList(
                [SparkleConstraint(generator.generate, num_beams, forced_bos)]),
            num_beams=num_beams,
            num_return_sequences=num_beams,
            max_length=MAX_LENGTH,
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)


# the README's quickstart example and the output the README gives for it (7 beams)
README_QUESTION = "Name the movies distributed by Warner Bros. and directed by Ridley Scott."
README_OUTPUT = ("SELECT DISTINCT var_uri WHERE brack_open var_uri [ distributor : property ] "
                 "[ warner_bros. : resource ] sep_dot var_uri [ director : property ] "
                 "[ ridley_scott : resource ] sep_dot var_uri [ type : rdf ] "
                 "[ film : ontology ] brack_close")
got = " ".join(generate(README_QUESTION, 7)[0].split())
print(f"README example: {got}")
if got == README_OUTPUT:
    print("README example reproduced", flush=True)
else:
    print("WARNING: the README example was NOT reproduced -- SPARKLE may not be "
          f"running as published here. README says:\n{README_OUTPUT}", flush=True)

results = []
start = time.perf_counter()
for i, item in enumerate(data, 1):
    try:
        candidates = generate(item["question"], NUM_BEAMS)
    except Exception as e:  # one bad question must not lose the others
        candidates = [f"ERROR: {e}"]
    results.append({"question": item["question"], "gold": item["sparql_query"], "candidates": candidates})
    print(f"{i}/{len(data)}  {' '.join(candidates[0].split())}", flush=True)

out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{len(results)} questions in {time.perf_counter() - start:.0f}s -> {out_path}")

Overwriting /content/run_sparkle.py


In [8]:
!python /content/run_sparkle.py {SPARKLE_DIR} {SPARKLE_DATA}/lcquad1 {REPO_DIR} {OUT_DIR}/sparkle_lcquad1.json

/content/drive/MyDrive/ogd/counterfactuals/sparkle_lcquad1.json holds a different set of questions -- regenerating it
Loading weights: 100% 515/515 [00:00<00:00, 17082.24it/s]
loaded /content/sparkle_data/lcquad1/output on cuda:0, forced BOS token 0
loaded tries and link dictionaries in 12s
{'SELECT': 50283, 'ASK': 50284, 'VARS': [50281, 50282], 'WRAP_OPEN': 50265, 'WRAP_CLOSE': 50266, 'WHERE': 50267, 'COND_OPEN': 50268, 'COND_CLOSE': 50269, 'SEP_DOT': 50270, 'COUNT': 50271, 'ATTR_OPEN': 50272, 'ATTR_CLOSE': 50273, 'ORDER_BY': 50274, 'LIMIT': 50275, 'FILTER': 50276, 'EXISTS': 50277, 'NOT': 50278, 'OR': 50279, 'DESC': 50280}
100% 1884/1884 [00:48<00:00, 38.87it/s] 
built SPARKLE's decoder in 200s
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress 

### SPARKLE's drafts → SPARQL

SPARKLE's model writes each query as a draft in its own token form (`[ tahiti : resource ]` for an IRI, `brack_open` for `{`). Its `evaluate_sparkle.py` turns drafts into SPARQL with `LCQuAD1.postprocess`, looking each label up in `entity_relation_token_dict.pkl`. That file is 2 GB on disk and likely 13–15 GB once loaded, so this runs in its own process and needs the High-RAM runtime. If it runs out of memory, the comparison below still runs, on the drafts alone.

In [9]:
%%writefile /content/sparkle_to_sparql.py
"""SPARKLE's drafts -> SPARQL, by SPARKLE's own LCQuAD1.postprocess.

evaluate_sparkle.py builds two {label: IRI} lookups from the whole of
entity_relation_token_dict.pkl. Only the labels the drafts actually use are kept
here: those are the only ones postprocess() looks up, so the SPARQL is the same
as with the whole file, without holding two copies of it. Adds a "sparql" list
to SPARKLE's output file, one per candidate ("" where postprocess judges a draft
invalid).

    python sparkle_to_sparql.py <SPARKLE repo> <lcquad1 data dir> <SPARKLE output json>
"""
import json
import pickle
import re
import sys
import time
from pathlib import Path

sparkle_dir, data_dir, out_path = map(Path, sys.argv[1:4])
results = json.loads(out_path.read_text(encoding="utf-8"))
if all("sparql" in r for r in results):
    print(f"{out_path} already holds SPARQL -- nothing to do")
    sys.exit(0)

sys.path.insert(0, str(sparkle_dir))
from sparkle.dataset.lcquad1 import LCQuAD1  # noqa: E402

needed = {label for r in results for c in r["candidates"] for label in re.findall(r"\[[^\]]+\]", c)}

t0 = time.time()
with open(data_dir / "preprocessed" / "entity_relation_token_dict.pkl", "rb") as f:
    entity_token_dict, relation_token_dict = pickle.load(f)
print(f"loaded SPARKLE's vocabulary in {time.time() - t0:.0f}s", flush=True)

token_entity_dict = {v["decoded"]: k for k, v in entity_token_dict.items() if v["decoded"] in needed}
token_relation_dict = {v["decoded"]: k for k, v in relation_token_dict.items() if v["decoded"] in needed}
del entity_token_dict, relation_token_dict

for r in results:
    # postprocess replaces "brack_close " with its trailing space; decoding may
    # or may not keep that space, and one more is harmless
    r["sparql"] = LCQuAD1.postprocess(token_entity_dict, token_relation_dict,
                                      [c + " " for c in r["candidates"]])
out_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"SPARQL for {len(results)} questions -> {out_path}")

Overwriting /content/sparkle_to_sparql.py


In [10]:
!python /content/sparkle_to_sparql.py {SPARKLE_DIR} {SPARKLE_DATA}/lcquad1 {OUT_DIR}/sparkle_lcquad1.json

loaded SPARKLE's vocabulary in 32s
SPARQL for 23 questions -> /content/drive/MyDrive/ogd/counterfactuals/sparkle_lcquad1.json


## Comparison

Each output is scored against the counterfactual query:

- **Ours**: canonical string match, the same `canonicalize` as `test.py`, both exact and modulo namespace twins (`ontology/x` vs `property/x`). Twins count because the counterfactuals were checked to be empty under both namespaces.
- **SPARKLE**: its top-ranked candidate, compared in SPARKLE's own token form (`[ office : property ]` for an IRI) the way its `evaluate_sparkle.py` compares a draft with the gold query. Its own LC-QuAD files give the IRI ↔ label pairs. `sparkle_in_beam` says whether the counterfactual was anywhere among its 10 candidates.

SPARKLE's SPARQL (`sparkle_sparql`) is shown as well, but it is not what is scored. `postprocess` looks a label up among entities before relations, and an entity label is lower-cased while a relation label keeps its case. So a relation whose name is already all lower case shares its label with a class: `ontology/university` and the class `ontology/University` are both `[ university : ontology ]`. Its SPARQL then names the class. Two of the counterfactual relations are like this (president, ideology). Scoring the token form credits SPARKLE with the relation its model chose.

SPARKLE's own evaluation runs its candidates against an endpoint and keeps the first one with a non-empty answer, which a counterfactual never has. With no endpoint here, its top-ranked candidate is scored instead.

In [11]:
import json
import re
import sys

import pandas as pd

sys.path.insert(0, str(SPARKLE_DIR))
from sparkle.dataset.lcquad1 import LCQuAD1

ogd = json.loads((OUT_DIR / 'ogd_beam3_positive.json').read_text(encoding='utf-8'))
spk = json.loads((OUT_DIR / 'sparkle_lcquad1.json').read_text(encoding='utf-8'))
assert [r['question'] for r in ogd] == [r['question'] for r in spk], 'the two runs cover different questions'


# --- ours: canonicalize and the twin rewrite, copied from test.py (importing
# test.py would load the Qwen checkpoint); predicates.txt path adjusted
def canonicalize(q):
    q = " ".join(q.split())
    q = q.replace("COUNT( ?uri )", "COUNT(?uri)")
    q = q.replace("{", "{ ").replace("}", " }")
    q = re.sub(r"\s*\.\s*(?![^<]*>)", " . ", q)
    return " ".join(q.split()).replace(" . }", " }")


def _load_twin_names():
    ont, prop = set(), set()
    for line in (REPO_DIR / "lcquad_data" / "predicates.txt").read_text(encoding="utf-8").splitlines():
        iri = line.strip().rstrip(",")
        if "/ontology/" in iri:
            ont.add(iri.rsplit("/", 1)[1])
        elif "/property/" in iri:
            prop.add(iri.rsplit("/", 1)[1])
    return ont & prop


TWIN_NAMES = _load_twin_names()
TWIN_RE = re.compile(
    r"<http://dbpedia\.org/(?:ontology|property)/("
    + "|".join(sorted((re.escape(n) for n in TWIN_NAMES), key=len, reverse=True))
    + r")>"
)


def canonicalize_twins(q):
    return TWIN_RE.sub(r"<dbpedia-twin/\1>", canonicalize(q))


# --- SPARKLE: its preprocessed LC-QuAD files hold every query twice, as IRIs
# ("sparql") and in its token form ("sparql_proc"), in the same order, which
# gives the IRI <-> label pairs without loading its 2 GB vocabulary
PRE = SPARKLE_DATA / 'lcquad1' / 'preprocessed'
iri_to_item = {}
for name in ('test_data.json', 'train_eval_data.json'):
    for row in json.loads((PRE / name).read_text(encoding='utf-8')):
        iris = re.findall(r'<[^>]+>', row['sparql'])
        items = re.findall(r'\[[^\]]+\]', row['sparql_proc'])
        if len(iris) == len(items):
            iri_to_item.update(zip(iris, items))

# a relation label keeps its local name's case and says its namespace
SPARKLE_TWIN_RE = re.compile(
    r"\[ (" + "|".join(sorted((re.escape(n) for n in TWIN_NAMES), key=len, reverse=True))
    + r") : (?:ontology|property) \]"
)


def norm(s):
    return " ".join(s.split())


def to_sparkle(sparql):
    """A gold query in SPARKLE's token form: LCQuAD1.cleanse_questions, then the
    IRI -> label swap and var_ renaming of change_entity_relation_repr. None if
    one of its IRIs never occurs in SPARKLE's LC-QuAD files."""
    # cleanse_questions never reads self, so it is called on the class
    cleansed = LCQuAD1.cleanse_questions(
        None, [{'corrected_question': '', 'sparql_query': sparql, 'answers': []}])[0]['sparql']
    tokens = []
    for token in cleansed.split():
        if LCQuAD1.is_entity(token) or LCQuAD1.is_relation(token):
            if token not in iri_to_item:
                return None
            token = iri_to_item[token]
        tokens.append(token)
    return norm(re.sub(r"\?(\w+)", r"var_\1", " ".join(tokens)))


def sparkle_twins(s):
    return SPARKLE_TWIN_RE.sub(r"[ \1 : twin ]", s)


if not all('sparql' in s for s in spk):
    print('no SPARQL in the SPARKLE output -- sparkle_to_sparql.py did not finish; '
          'its SPARQL columns are left empty and the drafts are scored as usual')

rows = []
for o, s in zip(ogd, spk):
    gold = o['gold']
    gold_s = to_sparkle(gold)
    candidates = [norm(c) for c in s['candidates']]
    top = candidates[0]
    rows.append({
        'kind': 'ASK' if gold.lstrip().upper().startswith('ASK') else 'SELECT',
        'question': o['question'],
        'gold': canonicalize(gold),
        'ours': canonicalize(o['generated']),
        'gold_sparkle_form': gold_s,
        'sparkle': top,
        'sparkle_sparql': canonicalize(s['sparql'][0]) if s.get('sparql') else '',
        'ours_exact': canonicalize(o['generated']) == canonicalize(gold),
        'ours_mod_twins': canonicalize_twins(o['generated']) == canonicalize_twins(gold),
        'sparkle_exact': None if gold_s is None else top == gold_s,
        'sparkle_mod_twins': None if gold_s is None else sparkle_twins(top) == sparkle_twins(gold_s),
        'sparkle_in_beam': None if gold_s is None else gold_s in candidates,
    })
df = pd.DataFrame(rows)

unscorable = df[df.sparkle_exact.isna()]
if len(unscorable):
    print(f'{len(unscorable)} questions have an IRI missing from SPARKLE\'s LC-QuAD files; '
          'their SPARKLE columns are left empty:')
    print('\n'.join(unscorable.question))

METRICS = ['ours_exact', 'ours_mod_twins', 'sparkle_exact', 'sparkle_mod_twins', 'sparkle_in_beam']


def summarise(part):
    line = {'n': len(part)}
    for metric in METRICS:
        hits = part[metric].dropna().astype(bool)
        line[metric] = f'{hits.sum()}/{len(hits)} ({hits.mean():.0%})' if len(hits) else 'n/a'
    return line


summary = pd.DataFrame({kind: summarise(df[df.kind == kind]) for kind in sorted(df.kind.unique())}
                       | {'all': summarise(df)}).T
display(summary)

pd.set_option('display.max_colwidth', None)
display(df[['kind', 'question', 'gold', 'ours', 'ours_exact',
            'gold_sparkle_form', 'sparkle', 'sparkle_exact', 'sparkle_in_beam', 'sparkle_sparql']])

(OUT_DIR / 'counterfactual_results.json').write_text(
    df.to_json(orient='records', indent=2, force_ascii=False), encoding='utf-8')
print('saved', OUT_DIR / 'counterfactual_results.json')

,n,ours_exact,ours_mod_twins,sparkle_exact,sparkle_mod_twins,sparkle_in_beam
SELECT,23,14/23 (61%),17/23 (74%),0/23 (0%),0/23 (0%),0/23 (0%)
all,23,14/23 (61%),17/23 (74%),0/23 (0%),0/23 (0%),0/23 (0%)


,kind,question,gold,ours,ours_exact,gold_sparkle_form,sparkle,sparkle_exact,sparkle_in_beam,sparkle_sparql
0,SELECT,What is the route end of Roberto Clemente Bridge ?,SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Roberto_Clemente_Bridge> <http://dbpedia.org/ontology/routeEnd> ?uri },SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Roberto_Clemente_Bridge> <http://dbpedia.org/ontology/routeEnd> ?uri },True,SELECT DISTINCT var_uri WHERE brack_open [ roberto_clemente_bridge : resource ] [ routeEnd : ontology ] var_uri brack_close,SELECT DISTINCT var_uri WHERE brack_open [ roberto_clemente_bridge : resource ] [ river : ontology ] var_uri brack_close,False,False,SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Roberto_Clemente_Bridge> <http://dbpedia.org/ontology/River> ?uri }
1,SELECT,What is the nearest city of Birmingham and Oxford Junction Railway ?,SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Birmingham_and_Oxford_Junction_Railway> <http://dbpedia.org/ontology/nearestCity> ?uri },"SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Birmingham> <http://dbpedia.org/ontology/nearestCity> ?uri . <http://dbpedia.org/resource/Oxford_Junction,_Iowa> <http://dbpedia.org/ontology/nearestCity> ?uri }",False,SELECT DISTINCT var_uri WHERE brack_open [ birmingham_and_oxford_junction_railway : resource ] [ nearestCity : ontology ] var_uri brack_close,SELECT DISTINCT var_uri WHERE brack_open [ birmingham_and_oxford_junction_railway : resource ] [ routeEnd : ontology ] var_uri sep_dot var_uri [ type : rdf ] [ location : ontology ] brack_close,False,False,SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Birmingham_and_Oxford_Junction_Railway> <http://dbpedia.org/ontology/routeEnd> ?uri . ?uri <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://dbpedia.org/ontology/Location> }
2,SELECT,What is the ingredient of Coffee ?,SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Coffee> <http://dbpedia.org/ontology/ingredient> ?uri },SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Coffee> <http://dbpedia.org/ontology/ingredient> ?uri },True,SELECT DISTINCT var_uri WHERE brack_open [ coffee : resource ] [ ingredient : ontology ] var_uri brack_close,SELECT DISTINCT var_uri WHERE brack_open var_uri [ ingredient : ontology ] [ coffee : resource ] brack_close,False,False,SELECT DISTINCT ?uri WHERE { ?uri <http://dbpedia.org/ontology/ingredient> <http://dbpedia.org/resource/Coffee> }
3,SELECT,What is the training of Demis Roussos ?,SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Demis_Roussos> <http://dbpedia.org/ontology/training> ?uri },SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Demis_Roussos> <http://dbpedia.org/ontology/training> ?uri },True,SELECT DISTINCT var_uri WHERE brack_open [ demis_roussos : resource ] [ training : ontology ] var_uri brack_close,ASK WHERE brack_open [ demis_roussos : resource ] [ training : ontology ] [ university_of_new_york : resource ] brack_close,False,False,ASK WHERE { <http://dbpedia.org/resource/Demis_Roussos> <http://dbpedia.org/ontology/training> <http://dbpedia.org/resource/University_Of_New_York> }
4,SELECT,What is the debut team of Aishath Saffa ?,SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Aishath_Saffa> <http://dbpedia.org/ontology/debutTeam> ?uri },SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Aishath_Saffa> <http://dbpedia.org/property/debutteam> ?uri },False,SELECT DISTINCT var_uri WHERE brack_open [ aishath_saffa : resource ] [ debutTeam : ontology ] var_uri brack_close,SELECT DISTINCT var_uri WHERE brack_open [ aishath_saffa : resource ] [ nationalteam : property ] var_uri brack_close,False,False,SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Aishath_Saffa> <http://dbpedia.org/property/nationalteam> ?uri }
5,SELECT,Who is the narrator of Class (2016 TV series) ?,SELECT DISTINCT ?uri WHERE { <http://dbpedia.org/resource/Class_(2016_TV_series)> <http://dbpedia.org/ontology/

saved /content/drive/MyDrive/ogd/counterfactuals/counterfactual_results.json


In [12]:
# the question with each system's query as SPARQL: SPARKLE's is its top-ranked
# candidate through its own postprocess, or the draft if that step did not run
queries = [{'question': r.question, 'ours': r.ours, 'sparkle': r.sparkle_sparql or r.sparkle}
           for r in df.itertuples()]
(OUT_DIR / 'counterfactual_queries.json').write_text(
    json.dumps(queries, indent=2, ensure_ascii=False), encoding='utf-8')
print('saved', OUT_DIR / 'counterfactual_queries.json')

saved /content/drive/MyDrive/ogd/counterfactuals/counterfactual_queries.json
